# Phase 5 — TS.z01 파일럿 학습

**전제**: `phase5_ts_drive_preprocess.ipynb`으로 TS.z01 전처리 완료  
Drive `finetune_dataset/`에 train/val 이미지+라벨 + data.yaml 이미 존재

**실행**: 런타임 → GPU(L4) 설정 → **위에서 아래로 전부 실행**

| Step | 내용 | 예상 시간 |
|------|------|----------|
| 1 | 환경 설정 + Drive 마운트 | 30초 |
| 2 | 데이터 현황 확인 | 10초 |
| 3 | 학습 | L4: 2~4시간 |
| 4 | 검증 (mAP) | 5분 |
| 5 | A/B 비교 | 2분 |
| 6 | best.pt 안내 | - |

## Step 1 — 환경 설정 + Drive 마운트

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi

In [ ]:
from google.colab import drive
import os, yaml
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/LGHellovision/Project 02/Object Detection'
FINETUNE_DIR = f'{DRIVE_ROOT}/finetune_dataset'
RUNS_DIR     = f'{DRIVE_ROOT}/runs'

yaml_path = f'{FINETUNE_DIR}/data.yaml'
raw = Path(yaml_path).read_text(encoding='utf-8') if os.path.exists(yaml_path) else ''
data_cfg = yaml.safe_load(raw) if raw else None

if data_cfg is None or 'names' not in data_cfg:
    print('data.yaml 비어있음 — 라벨에서 직접 복구')

    src_dir = Path(f'{FINETUNE_DIR}/train/labels')
    all_files = list(src_dir.glob('*.txt'))
    total = len(all_files)
    print(f'라벨 파일: {total:,}개')

    class_ids = set()
    for i, f in enumerate(all_files):
        for line in f.read_text().strip().split('\n'):
            if line.strip():
                class_ids.add(int(line.split()[0]))
        if (i + 1) % 100 == 0:
            print(f'  {i+1:,}/{total:,} ({(i+1)/total*100:.1f}%)', flush=True)

    print(f'스캔 완료 — {len(class_ids)}개 클래스 ID')

    nc = max(class_ids) + 1 if class_ids else 0
    ALL_CLASSES = [f'class_{i}' for i in range(nc)]

    data_cfg = {
        'train': f'{FINETUNE_DIR}/train/images',
        'val':   f'{FINETUNE_DIR}/val/images',
        'nc':    nc,
        'names': ALL_CLASSES,
    }
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_cfg, f, allow_unicode=True, default_flow_style=False)
    print(f'data.yaml 재생성 완료 — nc={nc}')
else:
    ALL_CLASSES = data_cfg['names']

print(f'FINETUNE_DIR: {FINETUNE_DIR}')
print(f'RUNS_DIR:     {RUNS_DIR}')
print(f'클래스 수:    {len(ALL_CLASSES)}종')
print(f'상위 10개:    {ALL_CLASSES[:10]}')

In [ ]:
!pip install ultralytics -q
print('ultralytics 설치 완료')

## Step 2 — 데이터 현황 확인

train/val 이미지+라벨이 Drive에 있는지 확인합니다.

In [ ]:
import subprocess

def count_files(dir_path, ext):
    """ls + wc로 Drive 파일 빠르게 카운팅"""
    r = subprocess.run(
        ['bash', '-c', f'ls "{dir_path}"/*.{ext} 2>/dev/null | wc -l'],
        capture_output=True, text=True
    )
    return int(r.stdout.strip()) if r.stdout.strip() else 0

train_imgs = count_files(f'{FINETUNE_DIR}/train/images', 'jpg')
train_lbls = count_files(f'{FINETUNE_DIR}/train/labels', 'txt')
val_imgs   = count_files(f'{FINETUNE_DIR}/val/images', 'jpg')
val_lbls   = count_files(f'{FINETUNE_DIR}/val/labels', 'txt')

print('=== finetune_dataset 현황 ===')
print(f'train: {train_imgs:,}장 이미지 / {train_lbls:,}개 라벨')
print(f'val:   {val_imgs:,}장 이미지 / {val_lbls:,}개 라벨')
print(f'nc:    {len(ALL_CLASSES)}종')

if train_imgs == 0:
    raise RuntimeError('train 이미지 없음 — phase5_ts_drive_preprocess.ipynb 먼저 실행')
if val_imgs == 0:
    raise RuntimeError('val 이미지 없음 — preprocess 확인')

print(f'\ndata.yaml 경로:')
print(f'  train: {data_cfg["train"]}')
print(f'  val:   {data_cfg["val"]}')
print('\n✅ 데이터 준비 완료 — Step 3 학습 진행 가능')

## Step 3 — 파인튜닝 학습

> base: `yolo11s.pt` / epochs=100 / patience=20 (early stop)  
> L4 기준 2~4시간, A100 기준 1~2시간

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
print(f'학습 시작 — 클래스: {len(ALL_CLASSES)}종')

results = model.train(
    data=f'{FINETUNE_DIR}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=20,
    project=RUNS_DIR,
    name='korean_food_v1',
    exist_ok=True,
    save=True,
    save_period=10,
    conf=0.001,
    iou=0.45,
    max_det=100,
)

print(f'\n학습 완료 — best.pt: {RUNS_DIR}/korean_food_v1/weights/best.pt')

## Step 4 — 검증 (mAP@0.5)

> 목표: mAP@0.5 ≥ 0.60  
> 0.40~0.59 → 데이터 추가(z02~) 후 재학습  
> 0.40 미만 → 설정/클래스 구조 점검

In [ ]:
from ultralytics import YOLO
import pandas as pd

best_path = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'
model_ft = YOLO(best_path)
metrics = model_ft.val(data=f'{FINETUNE_DIR}/data.yaml', device=0, conf=0.5, iou=0.45)

print(f'mAP@0.5:   {metrics.box.map50:.4f}  (목표 >= 0.60)')
print(f'mAP@0.5:95: {metrics.box.map:.4f}')
print(f'Precision:  {metrics.box.mp:.4f}')
print(f'Recall:     {metrics.box.mr:.4f}')

# 광고 타겟 핵심 클래스 AP
KEY_CLASSES = [
    '굴비', '조기', '대게', '전복', '장어', '한우', '흑돼지',
    '비빔밥', '김치찌개', '된장찌개', '삼겹살', '불고기',
    '사과', '딸기', '수박', '감귤',
]

if hasattr(metrics.box, 'ap_class_index'):
    rows = []
    for i, ci in enumerate(metrics.box.ap_class_index):
        nm = ALL_CLASSES[int(ci)] if int(ci) < len(ALL_CLASSES) else str(ci)
        if nm in KEY_CLASSES:
            rows.append({'클래스': nm, 'AP@0.5': round(float(metrics.box.ap50[i]), 4)})
    if rows:
        print('\n--- 핵심 클래스 AP ---')
        print(pd.DataFrame(rows).sort_values('AP@0.5', ascending=False).to_string(index=False))

print('\n✅ 합격 — 전체 데이터 풀 학습 진행 가능' if metrics.box.map50 >= 0.60
      else '\n⚠️  미달 — 데이터 추가 또는 설정 조정 필요')

## Step 5 — A/B 비교 (기존 yolo11s vs 파인튜닝)

val 이미지 20장으로 기존 모델 vs 파인튜닝 모델 탐지 건수 비교

In [ ]:
from ultralytics import YOLO
import glob
from collections import Counter

val_img_list = glob.glob(f'{FINETUNE_DIR}/val/images/*.jpg')[:20]
if not val_img_list:
    print('val 이미지 없음')
else:
    base = YOLO('yolo11s.pt')
    ft   = YOLO(f'{RUNS_DIR}/korean_food_v1/weights/best.pt')

    bc, fc_cnt = Counter(), Counter()
    for img in val_img_list:
        for p in base(img, conf=0.5, verbose=False):
            for b in p.boxes:
                bc[p.names[int(b.cls)]] += 1
        for p in ft(img, conf=0.5, verbose=False):
            for b in p.boxes:
                fc_cnt[p.names[int(b.cls)]] += 1

    print('=== 기존 yolo11s.pt 상위 10 ===')
    for label, n in bc.most_common(10):
        print(f'  {label}: {n}')

    print(f'\n=== 파인튜닝 best.pt 상위 10 ===')
    for label, n in fc_cnt.most_common(10):
        print(f'  {label}: {n}')

    KEY_SET = set(KEY_CLASSES)
    bk = sum(n for l, n in bc.items() if l in KEY_SET)
    fk = sum(n for l, n in fc_cnt.items() if l in KEY_SET)
    print(f'\n한식 핵심 클래스 탐지: 기존 {bk}건 → 파인튜닝 {fk}건 ({"+" if fk >= bk else ""}{fk - bk})')
    print('✅ 효과 확인' if fk > bk else '⚠️  효과 미미 — 데이터 추가 검토')

## Step 6 — best.pt 로컬 적용 안내

In [ ]:
import os

best = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'
if os.path.exists(best):
    size_mb = os.path.getsize(best) / 1024 / 1024
    print(f'best.pt — {size_mb:.1f} MB')
    print()
    print('로컬 적용 방법:')
    print('─' * 40)
    print(f'1. Drive에서 best.pt 다운로드')
    print(f'   경로: {best}')
    print()
    print('2. 로컬 저장')
    print('   Object_Detection/models/korean_food_v1_best.pt')
    print()
    print('3. config/detection_config.yaml 수정')
    print('   name: models/korean_food_v1_best.pt')
    print()
    print('4. 테스트 실행')
    print('   python -m pytest tests/test_phase1_setup.py -v')
    print()
    print('models/*.pt 는 .gitignore 대상 — 커밋 금지')
else:
    print('best.pt 없음 — Step 3 학습 완료 후 재실행')